In [ ]:
# -*- coding: utf-8 -*-
"""04_gradcam_visualization_improved.ipynb

Complete Improved Pap Smear Classification with Grad-CAM Visualization
"""

# ============================================
# CELL 1: Import Required Libraries
# ============================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from pathlib import Path
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Dense, Dropout, GlobalAveragePooling2D, 
                                      BatchNormalization, Input, Rescaling)
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import callbacks, optimizers
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)
set_seed()

print(f"TensorFlow version: {tf.__version__}")
print("Libraries imported successfully!")

# ============================================
# CELL 2: Configuration
# ============================================

# Paths
DATA_DIR = "../dataset"  # Update this to your dataset path
MODEL_SAVE_PATH = "../models/improved_papsmear_model.keras"
WEIGHTS_SAVE_PATH = "../models/improved_weights.weights.h5"
GRADCAM_OUTPUT_DIR = "../gradcam"

# Training parameters
IMG_SIZE = 384  # Higher resolution for better detail
BATCH_SIZE = 16
EPOCHS_PER_STAGE = 15
NUM_CLASSES = 5
LEARNING_RATES = [1e-3, 1e-4, 1e-5]  # For progressive unfreezing

# Class names
CLASS_NAMES = ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial-Intermediate']

print("Configuration loaded!")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Classes: {CLASS_NAMES}")

# ============================================
# CELL 3: Data Loading Function
# ============================================

def load_dataset(data_dir, img_size=384, test_size=0.2, val_size=0.1):
    """Load dataset from directory structure"""
    
    class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
    
    images = []
    labels = []
    
    print("Loading dataset...")
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"  Warning: {class_dir} not found")
            continue
            
        count = 0
        for img_file in os.listdir(class_dir):
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                img_path = os.path.join(class_dir, img_file)
                try:
                    img = image.load_img(img_path, target_size=(img_size, img_size))
                    img_array = image.img_to_array(img)
                    images.append(img_array)
                    labels.append(class_to_idx[class_name])
                    count += 1
                except Exception as e:
                    print(f"    Error loading {img_file}: {e}")
        print(f"  Loaded {count} images from {class_name}")
    
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels)
    
    print(f"\nTotal: {len(images)} images loaded")
    
    # Split data
    X_temp, X_test, y_temp, y_test = train_test_split(
        images, labels, test_size=test_size, stratify=labels, random_state=42
    )
    
    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=42
    )
    
    # Convert to categorical
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=NUM_CLASSES)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=NUM_CLASSES)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=NUM_CLASSES)
    
    print(f"\nSplit sizes:")
    print(f"  Train: {len(X_train)}")
    print(f"  Validation: {len(X_val)}")
    print(f"  Test: {len(X_test)}")
    
    # Print class distribution
    print("\nClass distribution in training set:")
    for i, name in enumerate(CLASS_NAMES):
        count = np.sum(y_train == i)
        print(f"  {name}: {count} ({count/len(y_train)*100:.1f}%)")
    
    return (X_train, y_train_cat, y_train), (X_val, y_val_cat, y_val), (X_test, y_test_cat, y_test)

# ============================================
# CELL 4: Build Improved Model
# ============================================

def build_improved_model(img_size=384, num_classes=5):
    """Build improved model with EfficientNetB3 backbone"""
    
    inputs = Input(shape=(img_size, img_size, 3))
    
    # Normalization
    normalized = Rescaling(1./255)(inputs)
    
    # Backbone - EfficientNetB3 with pretrained weights
    backbone = EfficientNetB3(
        weights='imagenet', 
        include_top=False, 
        input_tensor=normalized
    )
    backbone.trainable = False  # Initially frozen
    
    x = backbone.output
    
    # Global average pooling
    x = GlobalAveragePooling2D()(x)
    
    # Classification head with BatchNormalization and Dropout
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    return model, backbone

# ============================================
# CELL 5: Create Data Generators with Augmentation
# ============================================

def create_data_generators():
    """Create data generators with augmentation for training"""
    
    train_datagen = ImageDataGenerator(
        rotation_range=90,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=[0.85, 1.15],
        shear_range=0.15,
        brightness_range=[0.8, 1.2],
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='reflect'
    )
    
    val_datagen = ImageDataGenerator()
    
    return train_datagen, val_datagen

# ============================================
# CELL 6: Progressive Training
# ============================================

def train_progressive(model, backbone, X_train, y_train, X_val, y_val, 
                      epochs_per_stage=15, batch_size=16):
    """Train with progressive unfreezing"""
    
    train_datagen, val_datagen = create_data_generators()
    
    train_flow = train_datagen.flow(X_train, y_train, batch_size=batch_size, shuffle=True)
    val_flow = val_datagen.flow(X_val, y_val, batch_size=batch_size, shuffle=False)
    
    histories = []
    
    # Stage 1: Train only classification head
    print("\n" + "="*50)
    print("STAGE 1: Training classification head only")
    print("="*50)
    
    backbone.trainable = False
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy', 'auc']
    )
    
    history1 = model.fit(
        train_flow,
        validation_data=val_flow,
        epochs=epochs_per_stage,
        callbacks=[
            callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
        ],
        verbose=1
    )
    histories.append(history1)
    
    # Stage 2: Unfreeze last third of backbone
    print("\n" + "="*50)
    print("STAGE 2: Fine-tuning last third of backbone")
    print("="*50)
    
    backbone.trainable = True
    total_layers = len(backbone.layers)
    freeze_until = int(total_layers * 0.7)
    
    for i, layer in enumerate(backbone.layers):
        layer.trainable = (i >= freeze_until)
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy', 'auc']
    )
    
    history2 = model.fit(
        train_flow,
        validation_data=val_flow,
        epochs=epochs_per_stage,
        callbacks=[
            callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
        ],
        verbose=1
    )
    histories.append(history2)
    
    # Stage 3: Full fine-tuning
    print("\n" + "="*50)
    print("STAGE 3: Full fine-tuning")
    print("="*50)
    
    for layer in backbone.layers:
        layer.trainable = True
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy', 'auc']
    )
    
    history3 = model.fit(
        train_flow,
        validation_data=val_flow,
        epochs=epochs_per_stage,
        callbacks=[
            callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
        ],
        verbose=1
    )
    histories.append(history3)
    
    return histories

# ============================================
# CELL 7: Evaluation Functions
# ============================================

def evaluate_model(model, X_test, y_test_cat, y_test):
    """Comprehensive model evaluation"""
    
    # Normalize test data
    X_test_norm = X_test / 255.0
    
    # Predictions
    y_pred_proba = model.predict(X_test_norm)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nTest Accuracy: {accuracy:.4f}")
    
    # Classification report
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    
    # ROC-AUC
    print("\n" + "="*60)
    print("AUC SCORES")
    print("="*60)
    auc_scores = []
    for i in range(NUM_CLASSES):
        if np.sum(y_test == i) > 0:
            auc = roc_auc_score(y_test == i, y_pred_proba[:, i])
            auc_scores.append(auc)
            print(f"{CLASS_NAMES[i]}: {auc:.3f}")
    
    if auc_scores:
        print(f"\nMean AUC: {np.mean(auc_scores):.3f}")
    
    # Per-class metrics for Parabasal and Dyskeratotic
    print("\n" + "="*60)
    print("CRITICAL CLASSES ANALYSIS")
    print("="*60)
    
    para_idx = CLASS_NAMES.index('Parabasal')
    dys_idx = CLASS_NAMES.index('Dyskeratotic')
    
    for idx, name in [(para_idx, 'Parabasal'), (dys_idx, 'Dyskeratotic')]:
        tp = cm[idx, idx]
        fp = cm[:, idx].sum() - tp
        fn = cm[idx, :].sum() - tp
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\n{name}:")
        print(f"  Accuracy: {tp}/{np.sum(y_test == idx)} = {tp/np.sum(y_test == idx):.2%}" if np.sum(y_test == idx) > 0 else "  No test samples")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall: {recall:.3f}")
        print(f"  F1-Score: {f1:.3f}")
    
    return y_pred, y_pred_proba

# ============================================
# CELL 8: Plot Training History
# ============================================

def plot_training_history(histories):
    """Plot training history"""
    
    plt.figure(figsize=(15, 5))
    
    # Loss
    plt.subplot(1, 3, 1)
    for i, history in enumerate(histories):
        plt.plot(history.history['loss'], label=f'Stage {i+1} Train')
        plt.plot(history.history['val_loss'], label=f'Stage {i+1} Val', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy
    plt.subplot(1, 3, 2)
    for i, history in enumerate(histories):
        plt.plot(history.history['accuracy'], label=f'Stage {i+1} Train')
        plt.plot(history.history['val_accuracy'], label=f'Stage {i+1} Val', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    
    # AUC
    plt.subplot(1, 3, 3)
    for i, history in enumerate(histories):
        if 'auc' in history.history and 'val_auc' in history.history:
            plt.plot(history.history['auc'], label=f'Stage {i+1} Train')
            plt.plot(history.history['val_auc'], label=f'Stage {i+1} Val', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.title('Training and Validation AUC')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()

# ============================================
# CELL 9: Grad-CAM Visualization Class
# ============================================

class GradCAM:
    """Grad-CAM for model interpretation"""
    
    def __init__(self, model, base_model, last_conv_name='top_conv'):
        self.model = model
        self.base_model = base_model
        self.last_conv = base_model.get_layer(last_conv_name)
        self.grad_model = self._create_grad_model()
    
    def _create_grad_model(self):
        """Create gradient model for Grad-CAM"""
        inputs = self.base_model.input
        conv_output = self.last_conv.output
        x = self.base_model.output
        for layer in self.model.layers[1:]:
            x = layer(x)
        return tf.keras.Model(inputs=inputs, outputs=[conv_output, x])
    
    def compute_heatmap(self, image_path, class_idx=None):
        """Compute Grad-CAM heatmap for an image"""
        
        # Load and preprocess image
        img = image.load_img(image_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_tensor = tf.convert_to_tensor(np.expand_dims(img_array, axis=0) / 255.0, dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            conv_outputs, preds = self.grad_model(img_tensor)
            if class_idx is None:
                class_idx = tf.argmax(preds[0])
            loss = preds[:, class_idx]
        
        grads = tape.gradient(loss, conv_outputs)
        
        if grads is None:
            return None, None
        
        # Global average pooling
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        
        # Weighted sum
        conv_outputs = conv_outputs[0]
        heatmap = tf.zeros(conv_outputs.shape[:2], dtype=tf.float32)
        for i in range(pooled_grads.shape[0]):
            heatmap += pooled_grads[i] * conv_outputs[:, :, i]
        
        # ReLU and normalize
        heatmap = tf.maximum(heatmap, 0)
        if tf.reduce_max(heatmap) > 0:
            heatmap = heatmap / tf.reduce_max(heatmap)
        
        return heatmap.numpy(), class_idx.numpy()
    
    def visualize(self, image_path, save_path=None):
        """Create Grad-CAM visualization"""
        
        # Compute heatmap
        heatmap, pred_class = self.compute_heatmap(image_path)
        
        if heatmap is None:
            print("Failed to compute Grad-CAM")
            return
        
        # Load images
        img = image.load_img(image_path, target_size=(224, 224))
        img_cv = cv2.imread(image_path)
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        h, w = img_cv.shape[:2]
        
        # Get prediction
        img_array = image.img_to_array(img)
        img_tensor = np.expand_dims(img_array, axis=0) / 255.0
        predictions = self.model.predict(img_tensor, verbose=0)
        
        # Resize heatmap
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        
        # Create overlay
        superimposed = (0.5 * heatmap_colored + 0.5 * img_cv).astype(np.uint8)
        
        # Display
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title(f"Original Image\nPred: {CLASS_NAMES[pred_class]} ({predictions[0][pred_class]:.2%})")
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(heatmap, cmap='jet')
        plt.title("Grad-CAM Heatmap")
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(superimposed)
        plt.title("Grad-CAM Overlay")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Save if path provided
        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            cv2.imwrite(save_path, cv2.cvtColor(superimposed, cv2.COLOR_RGB2BGR))
            print(f"Saved to {save_path}")
        
        # Print all predictions
        print("\n" + "="*50)
        print("PREDICTIONS")
        print("="*50)
        for i, (name, prob) in enumerate(zip(CLASS_NAMES, predictions[0])):
            marker = "➤" if i == pred_class else " "
            print(f"{marker} {name}: {prob:.4f} ({prob:.1%})")
        print("="*50)
        
        return superimposed, heatmap

# ============================================
# CELL 10: Compare Parabasal vs Dyskeratotic
# ============================================

def compare_classes(gradcam, image_path, class1_name='Parabasal', class2_name='Dyskeratotic'):
    """Compare Grad-CAM for two classes"""
    
    class1_idx = CLASS_NAMES.index(class1_name)
    class2_idx = CLASS_NAMES.index(class2_name)
    
    # Load image
    img = image.load_img(image_path, target_size=(224, 224))
    img_cv = cv2.imread(image_path)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    h, w = img_cv.shape[:2]
    
    # Compute heatmaps for both classes
    hm1, _ = gradcam.compute_heatmap(image_path, class1_idx)
    hm2, _ = gradcam.compute_heatmap(image_path, class2_idx)
    
    if hm1 is None or hm2 is None:
        print("Failed to compute Grad-CAM for one or both classes")
        return
    
    # Resize
    hm1_resized = cv2.resize(hm1, (w, h))
    hm2_resized = cv2.resize(hm2, (w, h))
    
    # Difference map
    diff = hm1_resized - hm2_resized
    
    # Statistics
    correlation = np.corrcoef(hm1.flatten(), hm2.flatten())[0, 1]
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Row 1: Individual heatmaps
    axes[0, 0].imshow(img)
    axes[0, 0].set_title(f"Original Image")
    axes[0, 0].axis('off')
    
    im1 = axes[0, 1].imshow(hm1_resized, cmap='jet', vmin=0, vmax=1)
    axes[0, 1].set_title(f"{class1_name} Heatmap")
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
    
    im2 = axes[0, 2].imshow(hm2_resized, cmap='jet', vmin=0, vmax=1)
    axes[0, 2].set_title(f"{class2_name} Heatmap")
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)
    
    # Row 2: Comparative visualizations
    im3 = axes[1, 0].imshow(diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, 0].set_title(f"Difference Map\n(Red={class1_name}, Blue={class2_name})\nCorrelation: {correlation:.3f}")
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)
    
    # Winner-takes-all
    threshold = 0.1
    winner = np.zeros((h, w, 3), dtype=np.uint8)
    winner[diff > threshold] = [255, 80, 80]
    winner[diff < -threshold] = [80, 80, 255]
    winner[np.abs(diff) <= threshold] = [180, 180, 180]
    axes[1, 1].imshow(winner)
    axes[1, 1].set_title(f"Region Preference\n(Red={class1_name}, Blue={class2_name})")
    axes[1, 1].axis('off')
    
    # Overlay on original
    overlay = img_cv.copy()
    overlay[diff > threshold] = [255, 100, 100]
    overlay[diff < -threshold] = [100, 100, 255]
    axes[1, 2].imshow(overlay)
    axes[1, 2].set_title(f"Overlay on Original")
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('class_comparison.png', dpi=150)
    plt.show()
    
    # Print analysis
    print("\n" + "="*60)
    print("CLASS COMPARISON ANALYSIS")
    print("="*60)
    print(f"Correlation between heatmaps: {correlation:.3f}")
    
    if correlation > 0.7:
        print("  → HIGH CORRELATION: Model focuses on SIMILAR regions for both classes")
        print("  → This explains why the model confuses these classes")
    elif correlation < 0.3:
        print("  → LOW CORRELATION: Model focuses on DIFFERENT regions")
    else:
        print("  → MODERATE CORRELATION: Partial overlap in attention")
    
    # Region percentages
    total_pixels = h * w
    red_pct = 100 * (diff > threshold).sum() / total_pixels
    blue_pct = 100 * (diff < -threshold).sum() / total_pixels
    gray_pct = 100 - red_pct - blue_pct
    
    print(f"\nRegion preference (threshold={threshold}):")
    print(f"  Regions favoring {class1_name}: {red_pct:.1f}%")
    print(f"  Regions favoring {class2_name}: {blue_pct:.1f}%")
    print(f"  Ambiguous regions: {gray_pct:.1f}%")

# ============================================
# CELL 11: Main Training Pipeline
# ============================================

def main():
    """Main training pipeline"""
    
    print("="*60)
    print("IMPROVED PAP SMEAR CLASSIFICATION")
    print("="*60)
    
    # Check data directory
    if not os.path.exists(DATA_DIR):
        print(f"\nERROR: Data directory '{DATA_DIR}' not found!")
        print("Please update DATA_DIR to your dataset path.")
        return None, None
    
    # Load data
    print("\n[1/5] Loading dataset...")
    (X_train, y_train_cat, y_train), (X_val, y_val_cat, y_val), (X_test, y_test_cat, y_test) = load_dataset(DATA_DIR)
    
    if len(X_train) == 0:
        print("ERROR: No data loaded.")
        return None, None
    
    # Build model
    print("\n[2/5] Building model...")
    model, backbone = build_improved_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES)
    model.summary()
    
    # Train
    print("\n[3/5] Training model...")
    histories = train_progressive(
        model, backbone, X_train, y_train_cat, X_val, y_val_cat,
        epochs_per_stage=EPOCHS_PER_STAGE, batch_size=BATCH_SIZE
    )
    
    # Evaluate
    print("\n[4/5] Evaluating model...")
    y_pred, y_pred_proba = evaluate_model(model, X_test, y_test_cat, y_test)
    
    # Plot training history
    plot_training_history(histories)
    
    # Save model
    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
    model.save(MODEL_SAVE_PATH)
    model.save_weights(WEIGHTS_SAVE_PATH)
    print(f"\n✓ Model saved to {MODEL_SAVE_PATH}")
    print(f"✓ Weights saved to {WEIGHTS_SAVE_PATH}")
    
    # Final summary
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    
    final_val_acc = histories[-1].history['val_accuracy'][-1]
    final_test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\nFinal Validation Accuracy: {final_val_acc:.4f}")
    print(f"Final Test Accuracy: {final_test_acc:.4f}")
    
    return model, backbone

# ============================================
# CELL 12: Run Training (Optional - Skip if model already exists)
# ============================================

# Uncomment to train a new model
# model, backbone = main()

# Load existing model if already trained
if os.path.exists(MODEL_SAVE_PATH):
    print("Loading existing model...")
    model = load_model(MODEL_SAVE_PATH, compile=False)
    # Recreate backbone reference
    backbone = model.layers[0]
    print("Model loaded successfully!")
else:
    print("No existing model found. Training new model...")
    model, backbone = main()

# ============================================
# CELL 13: Grad-CAM Visualization on Test Image
# ============================================

# Initialize Grad-CAM
gradcam = GradCAM(model, backbone, last_conv_name='top_conv')

# Test image path
test_image_path = "../dataset/Parabasal/001.bmp"

# Create output directory
os.makedirs(GRADCAM_OUTPUT_DIR, exist_ok=True)

# Visualize
output_path = os.path.join(GRADCAM_OUTPUT_DIR, "gradcam_output.jpg")
superimposed, heatmap = gradcam.visualize(test_image_path, save_path=output_path)

# ============================================
# CELL 14: Compare Parabasal vs Dyskeratotic
# ============================================

# Compare the two confusing classes
compare_classes(gradcam, test_image_path, 'Parabasal', 'Dyskeratotic')

# ============================================
# CELL 15: Batch Analysis on Multiple Images
# ============================================

def batch_analysis(gradcam, data_dir, class_name, num_samples=5):
    """Analyze multiple images from a class"""
    
    class_dir = os.path.join(data_dir, class_name)
    if not os.path.exists(class_dir):
        print(f"Directory not found: {class_dir}")
        return
    
    images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    images = images[:num_samples]
    
    fig, axes = plt.subplots(2, num_samples, figsize=(4*num_samples, 8))
    
    for i, img_file in enumerate(images):
        img_path = os.path.join(class_dir, img_file)
        
        # Compute heatmap
        heatmap, pred_class = gradcam.compute_heatmap(img_path)
        
        if heatmap is not None:
            # Load original
            img = image.load_img(img_path, target_size=(224, 224))
            
            # Display
            axes[0, i].imshow(img)
            axes[0, i].set_title(f"{img_file[:15]}...\nPred: {CLASS_NAMES[pred_class]}")
            axes[0, i].axis('off')
            
            axes[1, i].imshow(heatmap, cmap='jet')
            axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'batch_analysis_{class_name}.png', dpi=150)
    plt.show()

# Run batch analysis (optional)
# batch_analysis(gradcam, DATA_DIR, 'Parabasal', num_samples=5)
# batch_analysis(gradcam, DATA_DIR, 'Dyskeratotic', num_samples=5)

# ============================================
# CELL 16: Generate Report
# ============================================

def generate_report(gradcam, image_path):
    """Generate a comprehensive analysis report"""
    
    print("\n" + "="*70)
    print("COMPREHENSIVE GRAD-CAM ANALYSIS REPORT")
    print("="*70)
    
    # Get prediction
    img_array = image.img_to_array(image.load_img(image_path, target_size=(224, 224)))
    img_tensor = np.expand_dims(img_array, axis=0) / 255.0
    predictions = gradcam.model.predict(img_tensor, verbose=0)
    
    print("\n1. MODEL PREDICTIONS")
    print("-" * 40)
    for i, (name, prob) in enumerate(zip(CLASS_NAMES, predictions[0])):
        print(f"   {name:25s}: {prob:.4f} ({prob:.1%})")
    
    print("\n2. CONFIDENCE ANALYSIS")
    print("-" * 40)
    top1_prob = np.max(predictions[0])
    top2_prob = np.sort(predictions[0])[-2]
    
    print(f"   Top-1 confidence: {top1_prob:.1%}")
    print(f"   Top-2 confidence: {top2_prob:.1%}")
    print(f"   Confidence gap: {(top1_prob - top2_prob):.1%}")
    
    if top1_prob < 0.5:
        print("   ⚠️  WARNING: Low confidence prediction")
    elif top1_prob < 0.7:
        print("   ⚠️  Moderate confidence - consider human review")
    else:
        print("   ✓ High confidence prediction")
    
    print("\n3. PARABASAL vs DYSKERATOTIC ANALYSIS")
    print("-" * 40)
    
    para_idx = CLASS_NAMES.index('Parabasal')
    dys_idx = CLASS_NAMES.index('Dyskeratotic')
    
    hm_para, _ = gradcam.compute_heatmap(image_path, para_idx)
    hm_dys, _ = gradcam.compute_heatmap(image_path, dys_idx)
    
    if hm_para is not None and hm_dys is not None:
        correlation = np.corrcoef(hm_para.flatten(), hm_dys.flatten())[0, 1]
        print(f"   Heatmap correlation: {correlation:.3f}")
        
        if correlation > 0.7:
            print("   ⚠️  HIGH CORRELATION: Model sees similar patterns for both classes")
            print("   → This explains classification difficulty")
        elif correlation < 0.3:
            print("   ✓ LOW CORRELATION: Model distinguishes well")
        else:
            print("   → Moderate correlation: Partial confusion")
    
    print("\n" + "="*70)
    print("END OF REPORT")
    print("="*70)

# Generate report
generate_report(gradcam, test_image_path)

print("\n" + "="*50)
print("All tasks completed successfully!")
print("="*50)

TensorFlow version: 2.18.0
Libraries imported successfully!
Configuration loaded!
Image size: 384x384
Classes: ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial-Intermediate']
No existing model found. Training new model...
IMPROVED PAP SMEAR CLASSIFICATION

[1/5] Loading dataset...
Loading dataset...
  Loaded 223 images from Dyskeratotic
  Loaded 238 images from Koilocytotic
  Loaded 271 images from Metaplastic
  Loaded 108 images from Parabasal
  Loaded 126 images from Superficial-Intermediate

Total: 966 images loaded

Split sizes:
  Train: 675
  Validation: 97
  Test: 194

Class distribution in training set:
  Dyskeratotic: 156 (23.1%)
  Koilocytotic: 166 (24.6%)
  Metaplastic: 190 (28.1%)
  Parabasal: 75 (11.1%)
  Superficial-Intermediate: 88 (13.0%)

[2/5] Building model...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 384, 384, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_3 (Rescaling)       │ (None, 384, 384, 3)       │               0 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_4 (Rescaling)       │ (None, 384, 384, 3)       │               0 │ rescaling_3[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalization_1               │ (None, 384, 384, 3)       │               7 │ rescaling_4[0][0]          │
│ (Normalization)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_5 (Rescaling)       │ (None, 384, 384, 3)       │               0 │ normalization_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv_pad (ZeroPadding2D) │ (None, 385, 385, 3)       │               0 │ rescaling_5[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv (Conv2D)            │ (None, 192, 192, 40)      │           1,080 │ stem_conv_pad[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_bn (BatchNormalization)  │ (None, 192, 192, 40)      │             160 │ stem_conv[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_activation (Activation)  │ (None, 192, 192, 40)      │               0 │ stem_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_dwconv                │ (None, 192, 192, 40)      │             360 │ stem_activation[0][0]      │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_bn                    │ (None, 192, 192, 40)      │             160 │ block1a_dwconv[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_activation            │ (None, 192, 192, 40)      │               0 │ block1a_bn[0][0]           │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_squeeze            │ (None, 40)                │               0 │ block1a_activation[0][0]   │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reshape (Reshape)  │ (None, 1, 1, 40)          │               0 │ block1a_se_squeeze[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reduce (Conv2D)    │ (None, 1, 1, 10)          │             41

 Total params: 11,738,932 (44.78 MB)

 Trainable params: 953,605 (3.64 MB)

 Non-trainable params: 10,785,327 (41.14 MB)


[3/5] Training model...

STAGE 1: Training classification head only
Epoch 1/15
